# PS04-AI — Bài làm sinh viên
**Nguyễn Bích Vân -IAI-UET-VNU**

Họ tên: …  
Mã sinh viên: …  
Lớp: …

Đề trong `PS04_AI_Exercises.pdf`. Bài 1–6 làm tay. Notebook này có mã sẵn cho hai bài kiểm chứng: chỉ cần bật cờ chạy hoặc thay tên tệp/cột. Chưa có đầu ra hay lời nhận xét điền sẵn.

## Khởi động
Mở đúng thư mục có `data/`, chọn kernel rồi chạy ô dưới. Thiếu thư viện: `%pip install -r requirements.txt` trong một ô Code mới, sau đó khởi động lại kernel.

In [ ]:
import sys
from pathlib import Path
print("Python của kernel:", sys.executable)
# Nếu thiếu thư viện: chạy %pip install -r requirements.txt trong một ô riêng,
# rồi Restart Kernel. Không dán dấu PS ...> hoặc >> vào notebook.
import numpy as np
import pandas as pd
import plotnine
import statsmodels
from IPython.display import display
from plotnine import (ggplot, aes, geom_point, geom_line, geom_abline,
                      geom_hline, labs, facet_wrap)
ROOT_OVERRIDE = ""  # Ví dụ r"D:\PS04_AI_Student"; để trống khi mở đúng thư mục.
ROOT = Path(ROOT_OVERRIDE).expanduser() if ROOT_OVERRIDE else Path.cwd()
ROOT = ROOT.resolve()
if not (ROOT / "regression_tools.py").is_file():
    raise FileNotFoundError("Mở thư mục có data/ và regression_tools.py, hoặc sửa ROOT_OVERRIDE.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from regression_tools import prepare_data, analyze, numeric_summary, make_plots, save_plots
print("Thư mục bài:", ROOT)
print("NumPy",np.__version__,"pandas",pd.__version__,"plotnine",plotnine.__version__,"statsmodels",statsmodels.__version__)


## Bài 1–6 — Phần tính tay
Đính kèm bài viết/PDF riêng; không dùng output notebook thay phép tính.

Câu khó nhất với tôi và điều cần hỏi giảng viên: …

## LAB01 — Câu 5 của Bài 7
Trước khi bật mã, trả lời câu 1–4 bằng lời. Sau đó chạy mô hình đơn và mô hình có nhóm trên cùng 54 lượt. Kiểm tra đơn vị một token, không nhầm với trăm token.

In [ ]:
RUN_LAB01 = False  # Doi thanh True sau khi da tra loi cau 1--4 cua Bai 7.
if RUN_LAB01:
    lab1 = prepare_data(ROOT,"exercise07_hardware_groups.csv",
                        "output_tokens","response_time_s","environment")
    result1 = analyze(lab1,reference="Edge")
    display(numeric_summary(result1))
    display(result1["simple"].params.to_frame("coefficient"))
    display(result1["adjusted"].params.to_frame("coefficient"))
    for group,d in lab1.groupby("group"):
        print(group,"n =",len(d),"r =",d.x.corr(d.y))
    figures1 = make_plots(result1,"Số token đầu ra","Thời gian phản hồi (giây)")
    for p in figures1.values():display(p)
    for path in save_plots(figures1,ROOT/"student_outputs","lab01"):print(path)
else:
    print("LAB01 chưa chạy. Sửa RUN_LAB01=True khi bắt đầu thực hành.")


### Bài 7 — Trả lời bằng lời
1. Bốn hệ số có thể đồng thời đúng không? …
2. Chọn hệ số cho câu hỏi trong cùng môi trường: …
3. Hai lỗi của câu kết luận: …
4. Biến cần thêm và vì sao: …
5. Hệ số trước/sau điều chỉnh, ý nghĩa và giới hạn: …

## Bài 8 — Cấp nhóm không phải cấp yêu cầu
A: …
B: …
C: …
D: …
Hai vấn đề cần phân biệt: …
Biến bổ sung đề xuất: …

## LAB02 — Câu 5 của Bài 9
Khớp mỗi chiều độc lập. Đừng lấy nghịch đảo slope làm lệnh thay cho khớp `x ~ y`.

In [ ]:
RUN_LAB02 = False  # Doi thanh True sau khi da du doan ket qua cua Bai 9.
if RUN_LAB02:
    data = prepare_data(ROOT,"exercise09_forward_reverse.csv",
                        "output_hundreds","response_time_s")
    result = analyze(data)
    CONFIG = {"x_label":"Độ dài đầu ra (trăm token)",
              "y_label":"Thời gian phản hồi (giây)"}
else:
    print("LAB02 chưa chạy. Sửa RUN_LAB02=True khi bắt đầu thực hành.")


In [ ]:
if RUN_LAB02:
    forward = result["simple"]
    reverse = result["reverse"]
    b_yx = float(forward.params["x"])
    b_xy = float(reverse.params["y"])
    print("Slope y theo x:",b_yx)
    print("Slope x theo y:",b_xy)
    print("Tích slope:",b_yx*b_xy,"; r^2 =",result["r"]**2)
    print("Tâm dữ liệu:",data.x.mean(),data.y.mean())
    # Chỉ đảo hệ số khác 0. Vẽ cả hai đường trên trục (x,y).
    grid = np.linspace(data.x.min(),data.x.max(),80)
    lines = pd.DataFrame({"x":grid,"y":forward.params["Intercept"]+b_yx*grid,
                          "fit":"OLS y on x"})
    if abs(b_xy) > 1e-12:
        second = pd.DataFrame({"x":grid,"y":(grid-reverse.params["Intercept"])/b_xy,
                               "fit":"OLS x on y"})
        lines = pd.concat([lines,second],ignore_index=True)
        p = (ggplot(data,aes("x","y"))+geom_point(size=2)
             +geom_line(aes(linetype="fit"),data=lines)
             +labs(x=CONFIG["x_label"],y=CONFIG["y_label"],linetype="Bài toán OLS"))
        display(p)
    else:
        print("Slope ngược bằng/gần 0; không chia cho nó để vẽ y theo x.")


### Bài 9 — Trả lời
1. Có phải cùng đường OLS không? …
2. Điều kiện cần và trường hợp r=0: …
3. Giải thích hình học: …
4. Tính từ thống kê tóm tắt và so sánh tại 14 giây: …
5. Các kiểm tra số học và hình đã tạo: …

## Bài 10 — Sửa các báo cáo
I: tên bẫy / câu sửa …
II: tên bẫy / câu sửa …
III: tên bẫy / câu sửa …
IV: tên bẫy / câu sửa …
Thiết kế dữ liệu bổ sung và điểm yếu được xử lý: …

## Tự kiểm tra trước khi nộp
Đã ghi tên/mã sinh viên; đã có phần làm tay; đã bật các LAB và lưu output; mỗi hình có nhãn; không diễn giải hệ số thành nhân quả; không có dữ liệu riêng hoặc đường dẫn máy cá nhân không cần thiết trong bài nộp.